In [ ]:
# Databricks notebook sourceMAGIC %md# Goal: Clean Members, normalize categories (Gender/Smoker), enforce FK to Policies, dedupe.## Steps### 1.Read Bronze### 2.Type casting & normalization### 3.Key check (Member_ID, Policy_ID) → quarantine nulls### 4.FK check (Policy_ID exists in Silver policies)### 5.Deduplicate by Member_ID (tie-breaking by Policy_ID)

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
#reading the data from bronzemem_bz = spark.read.format("delta").load(paths_bronze["members"])display(mem_bz)

In [ ]:
# ============ MEMBERS → SILVER (your existing hygiene) ============mem = (mem_bz.select(    F.col("Member_ID").cast("string"),    F.col("Policy_ID").cast("string"),    F.col("Age").cast("int"),    F.col("Gender").cast("string"),    F.col("BMI").cast("double"),    F.col("Smoker").cast("string"),    F.col("Chronic_Disease").cast("string"),    F.col("Employment_Status").cast("string"),    F.col("Region").cast("string")))# Normalize categories (Gender/Smoker) — your utilmem = normalize_categories(mem)# Key checkmem_bad = mem.filter(F.col("Member_ID").isNull() | F.col("Policy_ID").isNull())mem_ok  = mem.filter(F.col("Member_ID").isNotNull() & F.col("Policy_ID").isNotNull())if mem_bad.take(1):    quarantine(mem_bad, "null_key_member_or_policy", "members")# FK check to policies (existing Silver table)pol_keys = spark.table(f"{CATALOG_SILVER}.policies").select("Policy_ID").dropDuplicates()mem_ok = fk_filter(mem_ok, "Policy_ID", pol_keys, "Policy_ID", "members", "fk_policy_missing")# Optional imputationsif "Gender" in mem_ok.columns:    mem_ok = fill_with_mode(mem_ok, "Gender")# Deduplicate by Member_ID (keep latest by Policy_ID as tie-break)mem_silver = drop_dupes_keep_latest(mem_ok, ["Member_ID"], ["Policy_ID"])# ============ MEMBERS → SILVER+ (enterprise enrichment & audit) ============# 1) Feature engineering — keep it explainable & consistentmem_silver = (mem_silver    # Age bands    .withColumn("Age_Band",        F.when(F.col("Age") < 30, "18-29")         .when(F.col("Age") < 45, "30-44")         .when(F.col("Age") < 60, "45-59")         .otherwise("60+"))    # BMI category    .withColumn("BMI_Category",        F.when(F.col("BMI").isNull(), "Unknown")         .when(F.col("BMI") < 18.5, "Underweight")         .when(F.col("BMI") < 25, "Normal")         .when(F.col("BMI") < 30, "Overweight")         .otherwise("Obese"))    # Normalize Yes/No style for risk flags (works with your normalize_categories output)    .withColumn("Smoker_Norm",        F.when(F.lower("Smoker").isin("y","yes","1","true"), "Yes")         .when(F.lower("Smoker").isin("n","no","0","false"), "No")         .otherwise(F.col("Smoker")))    .withColumn("Chronic_Disease_Norm",        F.when(F.lower("Chronic_Disease").isin("y","yes","1","true"), "Yes")         .when(F.lower("Chronic_Disease").isin("n","no","0","false"), "No")         .otherwise(F.col("Chronic_Disease")))    # Simple lifestyle risk (0..1)    .withColumn("risk_smoker", F.when(F.col("Smoker_Norm") == "Yes", F.lit(1.0)).otherwise(F.lit(0.0)))    .withColumn("risk_chronic", F.when(F.col("Chronic_Disease_Norm") == "Yes", F.lit(1.0)).otherwise(F.lit(0.0)))    .withColumn("risk_bmi",        F.when(F.col("BMI").isNull(), 0.0)         .when(F.col("BMI") < 25, 0.0)         .when((F.col("BMI") >= 25) & (F.col("BMI") < 30), 0.3)         .otherwise(0.6))    .withColumn("Lifestyle_Risk_Score",        F.round(F.col("risk_smoker")*0.5 + F.col("risk_chronic")*0.3 + F.col("risk_bmi")*0.2, 2))    .drop("risk_smoker","risk_chronic","risk_bmi")    # Employment grouping    .withColumn("Employment_Group",        F.when(F.lower("Employment_Status").rlike("self|freelance|contract"), "Self-Employed")         .when(F.lower("Employment_Status").rlike("unemploy|student|retired"), "Not in Full-Time Work")         .when(F.lower("Employment_Status").rlike("full|part|employee|employed"), "Employed")         .otherwise("Other/Unknown"))    # Region normalization (light)    .withColumn("Region_Normalized", F.initcap("Region")))# 2) Member tenure from policies — earliest Policy_Start_Date per Policy_ID#    (Assumes your Silver policies table has Policy_ID + Policy_Start_Date)pol_silver = spark.table(f"{CATALOG_SILVER}.policies").select("Policy_ID", "Policy_Start_Date")first_start = pol_silver.groupBy("Policy_ID").agg(F.min("Policy_Start_Date").alias("First_Policy_Date"))mem_silver = (mem_silver    .join(first_start, on="Policy_ID", how="left")    .withColumn("Member_Tenure_Years",        F.round(F.datediff(F.current_date(), F.col("First_Policy_Date"))/365, 2))    .drop("First_Policy_Date"))# 3) Audit & data quality summary (uses your utils if you have them; otherwise comment out)mem_silver = add_audit_columns(mem_silver, "bronze.members")data_quality_summary(mem_silver, "members_silver")

In [ ]:
(mem_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(paths_silver["members"]))spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG_SILVER}")# Save DataFrames as tables in the databasemem_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG_SILVER}.members")print("Members → Silver complete.")

In [ ]:
members_df = spark.sql(f"SELECT * FROM {CATALOG_SILVER}.members")display(members_df)